# Experiment: Task 2 Colab Experiment: Scratch vs Finetune on Oxford-IIIT Pet

Objective:
- Compare `DenseNet121` and `ResNeXt50_32x4d` under two initialization strategies: `from scratch` and `fine-tune`.
- Use `Oxford-IIIT Pet` to make the report visually stronger than a CIFAR-style dataset.
- Produce four main runs, summary figures, and a LaTeX report source that can be compiled locally.

Success criteria:
- All four smoke tests finish.
- All four full runs finish and generate `history.csv`, `results.json`, `best.pt`, `curves.png`, `predictions_grid.png`, and `top_confusions.png`.
- The total recorded training time is at least 2 hours.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('This notebook is intended to run on Google Colab.')

REPO_URL = 'https://github.com/<your-username>/<your-repo>.git'
REPO_WEB_URL = REPO_URL.removesuffix('.git')
WORKSPACE = Path('/content/drive/MyDrive/task2_pet_workspace')
PROJECT_DIR = WORKSPACE / 'project'
DATA_DIR = WORKSPACE / 'data'
RUNS_DIR = WORKSPACE / 'runs'
SMOKE_DIR = WORKSPACE / 'smoke_runs'
REPORT_DIR = PROJECT_DIR / 'task2' / 'report'

def run(cmd: list[str], cwd: Path | None = None) -> None:
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

WORKSPACE.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    run(['git', 'clone', REPO_URL, str(PROJECT_DIR)])
else:
    run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'])

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'task2' / 'requirements-colab.txt')])
os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('RUNS_DIR =', RUNS_DIR)


In [ ]:
import torch
import torchvision

print('torch =', torch.__version__)
print('torchvision =', torchvision.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    run(['nvidia-smi'])
else:
    raise RuntimeError('Colab GPU is not enabled. Please switch Runtime -> Change runtime type -> GPU.')


## Plan

- Hypothesis 1: `fine-tune` should converge much faster than `scratch` for both architectures.
- Hypothesis 2: `DenseNet121` may offer a stronger speed/accuracy tradeoff on a free Colab GPU.
- Hypothesis 3: `ResNeXt50_32x4d` may become competitive only when the training budget is large enough.

Fixed outputs:
- Four main experiment folders under `RUNS_DIR`
- Summary charts and `summary.json` under `task2/report/`
- A generated `task2/report/main.tex` for final local PDF compilation


In [ ]:
PRESETS = [
    'densenet121_scratch',
    'densenet121_finetune',
    'resnext50_32x4d_scratch',
    'resnext50_32x4d_finetune',
]
DEVICE = 'cuda'
NUM_WORKERS = 2

print('Smoke-test presets:')
for preset in PRESETS:
    print(' -', preset)

print('\nFull run command:')
print(
    'python task2/run_all.py',
    '--data-root', DATA_DIR,
    '--output-root', RUNS_DIR,
    '--device', DEVICE,
    '--num-workers', NUM_WORKERS,
)


In [ ]:
# Smoke test: 1 epoch for all four settings
for preset in PRESETS:
    run([
        sys.executable,
        'task2/train.py',
        '--preset', preset,
        '--smoke-test',
        '--data-root', str(DATA_DIR),
        '--output-root', str(SMOKE_DIR),
        '--device', DEVICE,
        '--num-workers', str(NUM_WORKERS),
    ])

print('Smoke tests completed.')


In [ ]:
# Full run: four main experiments, then extend scratch runs until total train_seconds >= 7200
run([
    sys.executable,
    'task2/run_all.py',
    '--data-root', str(DATA_DIR),
    '--output-root', str(RUNS_DIR),
    '--device', DEVICE,
    '--num-workers', str(NUM_WORKERS),
])


In [ ]:
# Summarize figures and generate the local LaTeX source
run([
    sys.executable,
    'task2/summarize_runs.py',
    '--runs-root', str(RUNS_DIR),
    '--report-dir', str(REPORT_DIR),
])

run([
    sys.executable,
    'task2/make_report.py',
    '--summary-json', str(REPORT_DIR / 'summary.json'),
    '--outdir', str(REPORT_DIR),
    '--repo-url', REPO_WEB_URL,
])


## Results Snapshot

After the previous cell finishes, inspect the generated `summary.json` and preview the key figures below. If a fine-tune curve does not clearly rise faster than the matching scratch curve, first inspect the freeze/unfreeze logic and learning rates before changing the dataset or model family.


In [ ]:
import json
from IPython.display import display
from PIL import Image

summary = json.loads((REPORT_DIR / 'summary.json').read_text(encoding='utf-8'))
summary['rows']

for name in [
    'accuracy_comparison.png',
    'efficiency_tradeoff.png',
    'densenet_curves.png',
    'resnext_curves.png',
    'best_predictions_grid.png',
    'best_top_confusions.png',
]:
    image_path = REPORT_DIR / 'figs' / name
    if image_path.exists():
        print(name)
        display(Image.open(image_path))

print('Total recorded training seconds =', summary.get('total_train_seconds'))


In [ ]:
# Optional: package the report directory for download
import zipfile

bundle_path = WORKSPACE / 'task2_report_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in REPORT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(PROJECT_DIR))

print('Created:', bundle_path)
